# CyberNeura: AI Image Detection Model Training (CapCheck ViT & ResNet CNN)
### Google Colab GPU-Accelerated Training Notebook

This notebook fine-tunes **CapCheck AI Image Detector** (`aaronkantrowitz/ai-image-detection`) on the **GenImage / CIFAKE** dataset with generator-aware evaluation.

**Author:** CyberNeura AI Research & Security Team  
**Hardware:** NVIDIA T4 / V100 / A100 GPU

In [ ]:
# 1. Environment Setup & Google Drive Mount
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets evaluate accelerate torchvision scikit-learn pillow

In [ ]:
# 2. Check GPU Acceleration
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Active Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU is not active. Switch via Runtime -> Change runtime type.')

In [ ]:
# 3. Load CapCheck Model & Feature Extractor
from transformers import AutoImageProcessor, AutoModelForImageClassification

MODEL_ID = 'aaronkantrowitz/ai-image-detection'
print(f'Loading {MODEL_ID}...')
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0: 'AI_GENERATED', 1: 'REAL'},
    label2id={'AI_GENERATED': 0, 'REAL': 1},
    ignore_mismatched_sizes=True,
).to(device)

In [ ]:
# 4. Prepare Dataset Loader (GenImage / CIFAKE)
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader

DATA_DIR = '/content/drive/MyDrive/CyberNeura/archive'

class CyberNeuraImageDataset(Dataset):
    def __init__(self, file_tuples, processor):
        self.file_tuples = file_tuples
        self.processor = processor

    def __len__(self):
        return len(self.file_tuples)

    def __getitem__(self, idx):
        path, label = self.file_tuples[idx]
        with Image.open(path).convert('RGB') as img:
            inputs = self.processor(img, return_tensors='pt')
            item = {k: v.squeeze(0) for k, v in inputs.items()}
            item['labels'] = torch.tensor(label, dtype=torch.long)
            return item

# Indexing train & val folders
train_real = [(os.path.join(DATA_DIR, 'train/REAL', f), 1) for f in os.listdir(os.path.join(DATA_DIR, 'train/REAL'))[:5000]]
train_fake = [(os.path.join(DATA_DIR, 'train/FAKE', f), 0) for f in os.listdir(os.path.join(DATA_DIR, 'train/FAKE'))[:5000]]
train_samples = train_real + train_fake

val_real = [(os.path.join(DATA_DIR, 'test/REAL', f), 1) for f in os.listdir(os.path.join(DATA_DIR, 'test/REAL'))[:1000]]
val_fake = [(os.path.join(DATA_DIR, 'test/FAKE', f), 0) for f in os.listdir(os.path.join(DATA_DIR, 'test/FAKE'))[:1000]]
val_samples = val_real + val_fake

train_ds = CyberNeuraImageDataset(train_samples, image_processor)
val_ds = CyberNeuraImageDataset(val_samples, image_processor)
print(f'Datasets loaded: Train={len(train_ds)}, Val={len(val_ds)}')

In [ ]:
# 5. Training Arguments & Execution
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=preds, references=labels)['f1']
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir='./cyberneura-capcheck-checkpoints',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=torch.cuda.is_available(),
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

print('Starting ViT fine-tuning...')
trainer.train()

In [ ]:
# 6. Save Model Artifacts
SAVE_DIR = '/content/drive/MyDrive/CyberNeura/ml-service/models/image_model'
trainer.save_model(SAVE_DIR)
image_processor.save_pretrained(SAVE_DIR)
print(f'Model and processor successfully saved to {SAVE_DIR}')